&nbsp;

## ChIP-Atlas Diff Analysis Parse Notebook (Heat Shock Factor 1; HSF1)

- 既に ChIP-Atlas `diff Analysis` を実行済みの heat-stress related ChIP-seq dataset に対し、`bedtools intersect` を行い、その結果をparse するための notebook

In [1]:
import os
import datetime
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import plotly.express as px
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display


os.environ["POLARS_MAX_THREADS"] = "16"

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 1. Heat Shock Factor 1 (HSF1) motif Exploration

- `fanta.bio` の [CRE list FASTA file](../../Data/Database/fantabio/human-CREv1.2.1.hg38.cre-peaks.fa) を、MEME の `bed2fasta`で作成した後、JASPAR から、HSF1 の [motif data](https://jaspar.elixir.no/matrix/MA0486.1/) をダウンロードし、簡易的に、motif を調べておく
- 以下のコードで実行
- `cwltool --outdir ./out/fantabio/HSF1_motif/ ./Tools/fimo_motif.cwl`

In [2]:
HSF1_motif_scan_result = pl.read_csv(
    "../../out/fantabio/HSF1_motif/HSF1_MA0486_cre_peaks_fimo/fimo.tsv",
    separator="\t",
    n_threads=16,
    comment_prefix="#"
).filter(
    pl.col("motif_id").is_not_null()
).group_by(
    "sequence_name",
    maintain_order=True,
).agg(
    pl.struct(pl.exclude("sequence_name")).alias("motif_hit"),
    pl.len().alias("n_motif_hit")
).rename(
    {
        "sequence_name": "cre_id"
    }
)

display(HSF1_motif_scan_result)

cre_id,motif_hit,n_motif_hit
str,list[struct[9]],u32
"""FCHS_513416""","[{""MA0486.1"",""HSF1"",117,131,""+"",21.2535,9.3100e-10,0.0743,""CTTCTAGAAGGTTCT""}, {""MA0486.1"",""HSF1"",122,136,""-"",20.9296,4.6600e-9,0.0743,""CTTCTAGAACCTTCT""}, … {""MA0486.2"",""HSF1"",128,140,""+"",18.4677,5.3600e-7,0.206,""TTCTAGAAGCTTC""}]",6
"""FCHS_52690""","[{""MA0486.1"",""HSF1"",136,150,""+"",21.2535,9.3100e-10,0.0743,""CTTCTAGAAGGTTCT""}, {""MA0486.2"",""HSF1"",137,149,""+"",20.1129,1.9400e-7,0.206,""TTCTAGAAGGTTC""}, … {""MA0486.1"",""HSF1"",141,155,""-"",10.3662,0.0000618,0.501,""ACACTAGAACCTTCT""}]",4
"""FCHS_351829""","[{""MA0486.1"",""HSF1"",140,154,""+"",21.2535,9.3100e-10,0.0743,""CTTCTAGAAGGTTCT""}, {""MA0486.1"",""HSF1"",135,149,""-"",20.7887,8.3800e-9,0.0743,""CTTCTAGAAGCTTCC""}, … {""MA0486.2"",""HSF1"",136,148,""-"",18.4677,5.3600e-7,0.206,""TTCTAGAAGCTTC""}]",4
"""FCHS_105181""","[{""MA0486.1"",""HSF1"",392,406,""+"",21.2535,9.3100e-10,0.0743,""CTTCTAGAAGGTTCT""}, {""MA0486.2"",""HSF1"",393,405,""+"",20.1129,1.9400e-7,0.206,""TTCTAGAAGGTTC""}]",2
"""FCHS_409012""","[{""MA0486.1"",""HSF1"",57,71,""-"",21.1408,1.8600e-9,0.0743,""CTTCTAGAAGCTTCT""}, {""MA0486.2"",""HSF1"",58,70,""-"",18.4677,5.3600e-7,0.206,""TTCTAGAAGCTTC""}, … {""MA0486.1"",""HSF1"",125,139,""-"",12.2254,0.0000218,0.397,""GTTCTAGAATGCTCA""}]",5
…,…,…
"""FCHS_223475""","[{""MA0486.1"",""HSF1"",627,641,""-"",9.32394,0.0000998,0.544,""TTTGTGGAGGTTTCC""}]",1
"""FCHS_366858""","[{""MA0486.1"",""HSF1"",680,694,""-"",9.32394,0.0000998,0.544,""GTTGTGGAGCTTTCT""}]",1
"""FCHS_12001""","[{""MA0486.1"",""HSF1"",715,729,""-"",9.32394,0.0000998,0.544,""GTTCTAGATGTCTCA""}]",1


&nbsp;

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## 2. ChIP-Atlas Diff Analysis

&nbsp;

### この notebook を実行するために必要な手順

1. [ChIP-Atlas Diff Analysis](https://chip-atlas.org/diff_analysis) を実行する
    1. 事前に定義した heat-stress (dataset A) vs. ctrl (dataset B) のデータに従ってペアを作成し、実行する
2. `bedtools intersect -wo`を実行する CWLファイル [../../Workflow/bedtools_intersect_v2_scatter.cwl](../../Workflow/bedtools_intersect_v2_scatter.cwl)を実行する

In [3]:
INTERSECT_DIFF_COLUMNS = [
    # A: ChIP-Atlas diff DPR (8 cols)  <- query (-a)
    "dpr_chrom", "dpr_start", "dpr_end",
    "norm_count_A", "norm_count_B",  # dataset A → Heat stress, B → Control
    "log2fc", "p_value", "q_value",
    # B: fanta.bio CRE (11 cols)       <- database (-b)
    "cre_chrom", "cre_start", "cre_end", "cre_id",
    "cre_score", "cre_strand", "thickStart", "thickEnd",
    "itemRgb", "cre_name", "pooled_cre_scores",
    # -wo
    "overlap_bp",
]

In [4]:
#繰り返し処理を行う
BED_INTERSECT_DIR = pathlib.Path("../../out/ChIP-Atlas_diff/ChIPseq_hsf1/bedfile_intersect_v2/")

def read_intersect_bed(bed_path: pathlib.Path, q_threshold: float = 0.01) -> pl.DataFrame:
    """
    Read one bedtools intersect -wo output and tag with diff dataset ID from filename.
    default q-value threshold is 0.01.
    """
    # v2 filename: <dataset>_intersecting_<cre-peaks>.bed
    diff_dataset = bed_path.stem.split("_intersecting_", 1)[0]

    return (
        pl.read_csv(
            bed_path,
            separator="\t",
            has_header=False,
            new_columns=INTERSECT_DIFF_COLUMNS,
            # Force float dtypes: leading "1" q-values can make Polars infer Int64
            # and drop scientific-notation significant hits under ignore_errors.
            schema_overrides={
                "dpr_start": pl.Int64,
                "dpr_end": pl.Int64,
                "log2fc": pl.Float64,
                "p_value": pl.Float64,
                "q_value": pl.Float64,
                "cre_start": pl.Int64,
                "cre_end": pl.Int64,
                "cre_score": pl.Int64,
                "thickStart": pl.Int64,
                "thickEnd": pl.Int64,
                "overlap_bp": pl.Int64,
            },
            n_threads=16,
            ignore_errors=True,
        )
        .filter(pl.col("q_value") < q_threshold)
        .with_columns(
            # Fraction of CRE length covered by the DPR overlap (-wo)
            (pl.col("overlap_bp") / (pl.col("cre_end") - pl.col("cre_start"))).alias("overlap_frac_of_fantabio_cre"),
            pl.struct(
                "dpr_start",
                "dpr_end",
                "dpr_chrom"
            ).alias("dpr_interval")
        )
        .with_columns(
            pl.col("pooled_cre_scores").str.extract(r"directionality:([^|]+)", 1).cast(pl.Float64, strict=False).alias("directionality"),
            pl.col("pooled_cre_scores").str.extract(r"class:([^|]+)", 1).alias("fantabio_cre_class"),
        )
        .with_columns(
            pl.lit(diff_dataset).alias("diff_dataset"),
            pl.lit(bed_path.name).alias("source_bed_file"),
        )
        .drop("cre_score", "cre_strand", "thickStart", "thickEnd", "itemRgb", "pooled_cre_scores", "norm_count_A", "norm_count_B", "dpr_start", "dpr_end", "dpr_chrom")
    )

bed_paths = sorted(
    BED_INTERSECT_DIR.glob("*_intersecting_*.bed")
)

all_diff_df = (
    pl.concat(
        [read_intersect_bed(p) for p in bed_paths],
        how="vertical_relaxed",
    )
    .sort(
        "cre_chrom",
        descending=False,
    )
)


# mismatch = all_diff_df.filter(pl.col("cre_chrom") != pl.col("dpr_chrom"))
# print(f"mismatch rows: {mismatch.height}")
pl.Config.set_tbl_rows(10)
pl.Config.set_fmt_str_lengths(100)
display(all_diff_df)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.920226,0.002139,0.007216,"""chr1""",2474949,2475504,"""FCHS_1044""","""FCHS_1044""",191,0.344144,"{2475211,2475402,""chr1""}",0.296296,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
2.222658,0.000546,0.002169,"""chr1""",4670737,4670915,"""FCHS_447991""","""FCHS_447991""",178,1.0,"{4670693,4670915,""chr1""}",0.333333,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
3.983845,1.5985e-9,5.4329e-8,"""chr1""",11736866,11737342,"""FCHS_3616""","""FCHS_3616""",189,0.397059,"{11737153,11737368,""chr1""}",0.973568,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
3.983845,1.5985e-9,5.4329e-8,"""chr1""",11737227,11737538,"""FCHS_448629""","""FCHS_448629""",141,0.453376,"{11737153,11737368,""chr1""}",0.152778,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
3.573485,2.3186e-10,1.0554e-8,"""chr1""",12433676,12434259,"""FCHS_448761""","""FCHS_448761""",249,0.427101,"{12433863,12434112,""chr1""}",-0.594406,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-1.72397,2.9010e-7,0.000049,"""chrX""",103687974,103688485,"""FCHS_442436""","""cp3@MORF4L2,cp3@MORF4L2-AS1""",230,0.450098,"{103687887,103688204,""chrX""}",-0.996248,"""PLA""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-1.29167,0.00113,0.009301,"""chrX""",109536534,109537015,"""FCHS_442918""","""cp1@NXT2""",109,0.226611,"{109536398,109536643,""chrX""}",0.998494,"""PLA""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
-3.678532,0.00006,0.000405,"""chrY""",11329762,11329941,"""FCHS_447044""","""FCHS_447044""",16,0.089385,"{11329859,11329875,""chrY""}",-0.884058,"""ELA""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

# Selection of Differencially Peaked Region (DPR)

&nbsp;

## (1) Filtering Condition

- `log2(Fold Change) > 0.5`

In [5]:
all_diff_up = all_diff_df.filter(
    pl.col("log2fc") > 0.5
)

display(all_diff_up)

log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str
1.920226,0.002139,0.007216,"""chr1""",2474949,2475504,"""FCHS_1044""","""FCHS_1044""",191,0.344144,"{2475211,2475402,""chr1""}",0.296296,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
2.222658,0.000546,0.002169,"""chr1""",4670737,4670915,"""FCHS_447991""","""FCHS_447991""",178,1.0,"{4670693,4670915,""chr1""}",0.333333,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
3.983845,1.5985e-9,5.4329e-8,"""chr1""",11736866,11737342,"""FCHS_3616""","""FCHS_3616""",189,0.397059,"{11737153,11737368,""chr1""}",0.973568,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
3.983845,1.5985e-9,5.4329e-8,"""chr1""",11737227,11737538,"""FCHS_448629""","""FCHS_448629""",141,0.453376,"{11737153,11737368,""chr1""}",0.152778,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
3.573485,2.3186e-10,1.0554e-8,"""chr1""",12433676,12434259,"""FCHS_448761""","""FCHS_448761""",249,0.427101,"{12433863,12434112,""chr1""}",-0.594406,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3.258323,1.9058e-7,0.000002,"""chrX""",153337053,153337419,"""FCHS_446031""","""FCHS_446031""",306,0.836066,"{153336990,153337359,""chrX""}",-0.882353,"""ELA""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
7.183269,0.001623,0.007579,"""chrX""",153950216,153950531,"""FCHS_446391""","""FCHS_446391""",77,0.244444,"{153950454,153950662,""chrX""}",-1.0,"""ELA""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""
1.588694,0.000325,0.003804,"""chrX""",19673487,19673866,"""FCHS_436260""","""FCHS_436260""",218,0.575198,"{19673419,19673705,""chrX""}",-0.317073,"""ELA""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"""


&nbsp;

&nbsp;

&nbsp;

## Filter by Overlap Fraction

- 対象は直前の `log2fc > 0.5` 通過後（`all_diff_up`）
- 同一 `diff_dataset` × `cre_id` で複数 DPR が重なる場合、`overlap_frac_of_fantabio_cre` が最大の行のみ残す
- `FIMO` で予測した motif hit を join


In [6]:
# Among upregulated DPRs (log2fc > 0.5), keep max CRE-cover fraction
# per (diff_dataset, cre_id). Ties keep multiple rows.

all_diff_up_filtered = all_diff_up.filter(
    pl.col("overlap_frac_of_fantabio_cre")
    == pl.col("overlap_frac_of_fantabio_cre").max().over("diff_dataset", "cre_id")
).join(
    HSF1_motif_scan_result,
    on=["cre_id"],
    how="left",
    coalesce=True
)


print(f"(all_diff_up {all_diff_up.height} -> {all_diff_up_filtered.height})")
display(all_diff_up_filtered)

(all_diff_up 2282 -> 2153)


log2fc,p_value,q_value,cre_chrom,cre_start,cre_end,cre_id,cre_name,overlap_bp,overlap_frac_of_fantabio_cre,dpr_interval,directionality,fantabio_cre_class,diff_dataset,source_bed_file,motif_hit,n_motif_hit
f64,f64,f64,str,i64,i64,str,str,i64,f64,struct[3],f64,str,str,str,list[struct[9]],u32
1.920226,0.002139,0.007216,"""chr1""",2474949,2475504,"""FCHS_1044""","""FCHS_1044""",191,0.344144,"{2475211,2475402,""chr1""}",0.296296,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",341,355,""+"",10.5352,0.0000569,0.494,""TGCCCGGAACCTTCC""}, {""MA0486.1"",""HSF1"",346,360,""-"",9.87324,0.0000779,0.521,""GGACTGGAAGGTTCC""}]",2
2.222658,0.000546,0.002169,"""chr1""",4670737,4670915,"""FCHS_447991""","""FCHS_447991""",178,1.0,"{4670693,4670915,""chr1""}",0.333333,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",66,80,""+"",18.9859,2.4200e-7,0.0754,""GTTCCAGAATGTTCT""}, {""MA0486.2"",""HSF1"",67,79,""+"",19.7903,2.8300e-7,0.206,""TTCCAGAATGTTC""}, … {""MA0486.2"",""HSF1"",57,69,""+"",11.2903,0.0000143,0.659,""TTCCAGAGTGTTC""}]",6
3.983845,1.5985e-9,5.4329e-8,"""chr1""",11736866,11737342,"""FCHS_3616""","""FCHS_3616""",189,0.397059,"{11737153,11737368,""chr1""}",0.973568,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",367,381,""+"",12.8169,0.0000152,0.357,""CTTCCTGAAGCTTCC""}, {""MA0486.2"",""HSF1"",368,380,""+"",7.54839,0.0000523,0.833,""TTCCTGAAGCTTC""}]",2
3.983845,1.5985e-9,5.4329e-8,"""chr1""",11737227,11737538,"""FCHS_448629""","""FCHS_448629""",141,0.453376,"{11737153,11737368,""chr1""}",0.152778,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",6,20,""+"",12.8169,0.0000152,0.357,""CTTCCTGAAGCTTCC""}, {""MA0486.2"",""HSF1"",7,19,""+"",7.54839,0.0000523,0.833,""TTCCTGAAGCTTC""}]",2
3.573485,2.3186e-10,1.0554e-8,"""chr1""",12433676,12434259,"""FCHS_448761""","""FCHS_448761""",249,0.427101,"{12433863,12434112,""chr1""}",-0.594406,"""ELA""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""","""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",338,352,""+"",15.8873,0.0000023,0.205,""TTTCCAGCACATTCT""}, {""MA0486.2"",""HSF1"",339,351,""+"",12.4839,0.000008,0.6,""TTCCAGCACATTC""}]",2
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3.258323,1.9058e-7,0.000002,"""chrX""",153337053,153337419,"""FCHS_446031""","""FCHS_446031""",306,0.836066,"{153336990,153337359,""chrX""}",-0.882353,"""ELA""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",118,132,""+"",20.0563,6.3300e-8,0.0743,""CTTCTGGAACCTTCC""}, {""MA0486.1"",""HSF1"",113,127,""-"",19.2254,1.9700e-7,0.0743,""GTTCCAGAAGCTTCC""}, … {""MA0486.2"",""HSF1"",124,136,""-"",11.8548,0.000011,0.628,""CTCTGGAAGGTTC""}]",6
7.183269,0.001623,0.007579,"""chrX""",153950216,153950531,"""FCHS_446391""","""FCHS_446391""",77,0.244444,"{153950454,153950662,""chrX""}",-1.0,"""ELA""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""","""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""",null,null
1.588694,0.000325,0.003804,"""chrX""",19673487,19673866,"""FCHS_436260""","""FCHS_436260""",218,0.575198,"{19673419,19673705,""chrX""}",-0.317073,"""ELA""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""","""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""","[{""MA0486.1"",""HSF1"",79,93,""+"",9.59155,0.0000886,0.533,""CTACTAGAAGTTCCT""}]",1


In [12]:
# Aggregate upregulated + max-overlap DPRs to CRE level
all_diff_up_select = all_diff_up_filtered.group_by(
    [
        "cre_chrom",
        "cre_start",
        "cre_end",
        "cre_id",
        "cre_name",
        # "dpr_chrom",
        "directionality",
        "fantabio_cre_class",
        "motif_hit",
        "n_motif_hit"
    ]
).agg(
    pl.col("overlap_frac_of_fantabio_cre").alias("overlap_frac_of_fantabio_cre"),
    pl.col("dpr_interval").alias("dpr_interval"),
    pl.col("log2fc").alias("log2fc"),
    pl.col("p_value").alias("p_value"),
    pl.col("q_value").alias("q_value"),
    pl.col("diff_dataset").unique().alias("diff_dataset"),
    pl.col("diff_dataset").unique().count().alias("diff_dataset_count"),
    pl.col("source_bed_file").alias("source_bed_file"),
).sort(
    by=["diff_dataset_count"],
    descending=True
)

pl.Config.set_tbl_rows(10)
display(all_diff_up_select)

cre_chrom,cre_start,cre_end,cre_id,cre_name,directionality,fantabio_cre_class,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,str,str,f64,str,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""chr16""",89161193,89161743,"""FCHS_500244""","""FCHS_500244""",-0.185185,"""ELA""","[{""MA0486.1"",""HSF1"",340,354,""+"",20.1831,4.7500e-8,0.0743,""GTTCTGGAAGGTTCT""}, {""MA0486.1"",""HSF1"",371,385,""+"",19.338,1.8100e-7,0.0743,""GTTCCAGAAGGTTCC""}, … {""MA0486.2"",""HSF1"",305,317,""-"",9.56452,0.0000282,0.711,""TTCCGGAACATGC""}]",22,"[0.225455, 0.467273, … 0.670909]","[{89161421,89161545,""chr16""}, {89161331,89161588,""chr16""}, … {89161307,89161676,""chr16""}]","[4.543278, 4.390846, … 1.365859]","[6.0646e-13, 6.8653e-44, … 0.000847]","[9.6617e-11, 1.0943e-40, … 0.007484]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min"", ""pair2_GSE43579_hsf1_k562_hs_42C_30min"", … ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""]",5,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair2_GSE43579_hsf1_k562_hs_42C_30min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", … ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr1""",229895344,229895929,"""FCHS_459597""","""FCHS_459597""",-0.565217,"""ELA""","[{""MA0486.1"",""HSF1"",465,479,""+"",15.6197,0.000003,0.22,""GTTCCAGATTGTTCT""}, {""MA0486.2"",""HSF1"",466,478,""+"",14.0323,0.000004,0.458,""TTCCAGATTGTTC""}, … {""MA0486.2"",""HSF1"",356,368,""+"",6.56452,0.000071,0.901,""TTCTAGGTTGTTC""}]",6,"[0.452991, 0.220513, … 0.822222]","[{229895569,229895834,""chr1""}, {229895694,229895823,""chr1""}, … {229895448,229895929,""chr1""}]","[5.088039, 3.114993, … 3.113387]","[2.4364e-26, 3.2023e-12, … 8.0732e-12]","[6.2105e-23, 2.8358e-10, … 7.3305e-9]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min"", ""pair2_GSE43579_hsf1_k562_hs_42C_30min"", … ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""]",4,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair2_GSE43579_hsf1_k562_hs_42C_30min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", … ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr16""",88984079,88984680,"""FCHS_163247""","""FCHS_163247""",-0.355705,"""ELA""","[{""MA0486.2"",""HSF1"",210,222,""+"",22.0,2.9800e-8,0.188,""TTCTAGAACATTC""}, {""MA0486.1"",""HSF1"",209,223,""+"",19.7183,1.1400e-7,0.0743,""GTTCTAGAACATTCT""}, … {""MA0486.2"",""HSF1"",543,555,""+"",5.80645,0.000091,0.953,""ATCCGGAAGGATC""}]",13,"[0.455907, 0.086522, … 0.886855]","[{88984166,88984440,""chr16""}, {88984497,88984549,""chr16""}, … {88984147,88984760,""chr16""}]","[2.985148, 2.244147, … 1.611809]","[6.5307e-9, 0.000419, … 0.000043]","[1.6815e-7, 0.00903, … 0.001008]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min"", ""pair2_GSE43579_hsf1_k562_hs_42C_30min"", … ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min""]",4,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair2_GSE43579_hsf1_k562_hs_42C_30min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", … ""pair8_GSE270671_hsf1_pc3m_sint_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""chr19""",2679471,2679732,"""FCHS_511561""","""FCHS_511561""",-0.333333,"""ELA""","[{""MA0486.1"",""HSF1"",170,184,""+"",20.493,2.4200e-8,0.0743,""TTTCTAGAAGGTTCC""}, {""MA0486.1"",""HSF1"",175,189,""-"",20.4085,2.9800e-8,0.0743,""CTTCTGGAACCTTCT""}, … {""MA0486.2"",""HSF1"",151,163,""+"",5.59677,0.0000984,0.961,""CTCTAGAAGGCTC""}]",20,"[0.452107, 0.532567, … 0.94636]","[{2679614,2679802,""chr19""}, {2679550,2679689,""chr19""}, … {2679485,2679801,""chr19""}]","[8.309778, 3.251928, … 1.446591]","[5.3310e-14, 2.9689e-17, … 0.000377]","[1.5098e-11, 4.7324e-15, … 0.004192]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60

In [13]:
all_diff_up_select.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/only_cre/fantabio_cre_list_diff_analysis_chipseq_hsf1.parquet",
    compression="zstd",
    compression_level=9
)

&nbsp;

&nbsp;

&nbsp;

&nbsp;

## Intergration (upregulated transcripts)

In [14]:
# up5p transcript lists + CRE
fantabio_HN2_up5p_transcripts_cre = pl.read_parquet(
    "../../out/fantabio/parquet_file/up5p_transcript_fantabio_v261001.parquet"
).join(
    all_diff_up_select,
    on=["cre_id", "cre_chrom", "cre_name", "directionality", "fantabio_cre_class"],
    how="inner",
    coalesce=True
)

fantabio_HN2_up5p_transcripts_cre.write_parquet(
    "../../out/ChIP-Atlas_diff/diff_analysis_filter_parquet/transcript_cre/fantabio_HN2_up5p_v261008_diff_analysis_chipseq_hsf1.parquet"
)


print(f"up5p_fantabio transcript count: {fantabio_HN2_up5p_transcripts_cre.select('transcript_id').n_unique()}")
print(f"up5p_fantabio transcript-CRE pairs count: {fantabio_HN2_up5p_transcripts_cre.select(['transcript_id', 'cre_id']).n_unique()}")
pl.Config.set_tbl_rows(4)
display(fantabio_HN2_up5p_transcripts_cre)

up5p_fantabio transcript count: 1400
up5p_fantabio transcript-CRE pairs count: 2292


transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28105""","""chr1""",161523745,"""161524487""","""cp1@HSPA6""","[""GQ891451""]","""0""",53,"[{""EH38E2843853"",""pELS""}, {""EH38E2843855"",""PLS""}, {""EH38E3981533"",""pELS""}]","[{""STAG2"",4103,[{""SRX9975614"",4103}, {""SRX9975613"",3500}, {""SRX26159208"",562}]}, {""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX13845684"",554}]}, … {""GABPA"",500,[{""SRX2635792"",500}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]",null,null,0.982111,"""PLA""","""204,0,0""","""red""",161523744,161524487,"[{""MA0486.2"",""HSF1"",722,734,""+"",14.5323,0.000003,0.41,""TTCGCGAAAGTTC""}, {""MA0486.1"",""HSF1"",696,710,""-"",12.4225,0.0000193,0.383,""CTTCCCGCACCTTCC""}, … {""MA0486.2"",""HSF1"",717,729,""-"",7.74194,0.0000494,0.817,""TTCGCGAACCTTT""}]",8,"[0.363392, 0.54105]","[{161524217,161524537,""chr1""}, {161524085,161524630,""chr1""}]","[3.696223, 3.583503]","[1.2063e-25, 4.3546e-12]","[1.5375e-22, 1.0794e-10]","[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min"", ""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min""]",2,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed"", ""pair6_GSE209667_hsf1_polyclonal_ab_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000309758.6""",352,17,37,335,232,4,15,228,"""HSPA6""","""HGNC:5239""","""protein_coding""","""ENSG00000173110.8""","[""basic"", ""Ensembl_canonical"", … ""CCDS""]","""+""",6,null,null,null,"""ensembl_havana_gene_homo_sapiens""","[""MANE_select""]",1,"""FCHS_28106""","""chr1""",161524161,"""161524546""","""cp4@HSPA6""","[""ENST00000309758.6"", ""NM_002155.5"", … ""AK295212""]","""0""",0,"[{""EH38E2843855"",""PLS""}]","[{""HSF1"",4038,[{""SRX16646095"",4038}, {""SRX16646111"",3572}, … {""SRX3322007"",560}]}, {""CTCF"",3651,[{""SRX10975999"",3651}, {""SRX10976007"",2920}, … {""SRX13551280"",511}]}, … {""MED12"",511,[{""SRX10852993"",511}]}]","[{""HS_00013400"",""H1 BMP4 Derived Mesendoderm Cultured Cells"",""Mesendoderm cell"",""Others"",""SVM""}, {""HS_00013401"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}, … {""HS_00013403"",""Foreskin Fibroblast Primary Cells skin02"",""Prepuce"",""Skin"",""SVM""}]","[{""hg19::chr1:161494328..161494332,+;hg_6264.1"",""p1@HSPA6""}]",null,0.943234,"""PLA""","""204,0,0""","""red""",161524160,161524546,"[{""MA0486.2"",""HSF1"",306,318,""+"",14.5323,0.000003,0.41,""TTCGCGAAAGTTC""}, {""MA0486.1"",""HSF1"",280,294,""-"",12.4225,0.0000193,0.383,

In [15]:
fantabio_HN2_up5p_transcripts_cre_lncRNA = fantabio_HN2_up5p_transcripts_cre.filter(
    pl.col("transcript_type") == "lncRNA"
)

display(fantabio_HN2_up5p_transcripts_cre_lncRNA)

transcript_id,UP2,DOWN2,UNCHANGED2,HN2,UP2_stranded,DOWN2_stranded,UNCHANGED2_stranded,HN2_stranded,gene_symbol,HGNC_ID,transcript_type,gene_id,tags,strand,number_of_tags,RNACentral_ID,corrrespondence,polyA_features,gene_source,annotation_remarks,annotation_remark_count,cre_id,cre_chrom,cre_chrom_start,cre_chrom_end,cre_name,tss_nearest_transcript_ids,tss_nearest_transcript_distance,transcript_distance,screen_ccres,chip_atlas_antigens,silencerdbs,fantom5_cage_peaks,fantom5_enhancers,directionality,fantabio_cre_class,itemRgb,itemRgb_name,cre_start,cre_end,motif_hit,n_motif_hit,overlap_frac_of_fantabio_cre,dpr_interval,log2fc,p_value,q_value,diff_dataset,diff_dataset_count,source_bed_file
str,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,list[str],str,u32,str,str,list[str],str,list[str],u32,str,str,i64,str,str,list[str],str,i64,list[struct[2]],list[struct[3]],list[struct[5]],list[struct[2]],list[struct[1]],f64,str,str,str,i64,i64,list[struct[9]],u32,list[f64],list[struct[3]],list[f64],list[f64],list[f64],list[str],u32,list[str]
"""ENST00000850140.1""",317,6,83,311,207,2,42,205,null,null,"""lncRNA""","""ENSG00000310471.1""","[""basic"", ""Ensembl_canonical"", … ""TAGENE""]","""+""",4,"""URS0002A15385""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project"", ""overlaps pseudogene""]",2,"""FCHS_28129""","""chr1""",161605881,"""161606178""","""cp1@HSPA7""","[""NR_024151.2""]","""0""",0,"[{""EH38E2843898"",""pELS""}]","[{""CTCF"",1775,[{""SRX10975999"",1775}, {""SRX10976007"",1383}, … {""SRX10976004"",607}]}, {""BCOR"",952,[{""SRX10382399"",952}, {""SRX10382441"",932}]}, … {""PTEN"",504,[{""SRX5416007"",504}]}]","[{""HS_00013426"",""H1 Cells"",""Embryonic stem cell"",""Stem cell"",""SVM""}]","[{""hg19::chr1:161575959..161575964,+;hg_6268.1"",""p1@HSPA7""}]",null,1.0,"""ELA""","""255,202,74""","""yellow""",161605880,161606178,"[{""MA0486.1"",""HSF1"",214,228,""-"",12.4225,0.0000193,0.383,""CTTCCCGCACCTTCC""}, {""MA0486.2"",""HSF1"",31,43,""+"",9.69355,0.0000269,0.705,""AGCTAGAACCTTC""}, … {""MA0486.1"",""HSF1"",30,44,""+"",10.8732,0.0000478,0.478,""GAGCTAGAACCTTCC""}]",5,[0.395973],"[{161606060,161606234,""chr1""}]",[2.465951],[0.000006],[0.000045],"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min""]",1,"[""pair1_GSE38901_hsf1_mcf10a_hs_42C_60min_intersecting_human-CREv1.2.1.hg38.cre-peaks.bed""]"
"""ENST00000729765.1""",315,20,71,295,206,4,41,202,null,null,"""lncRNA""","""ENSG00000295391.1""","[""basic"", ""TAGENE""]","""+""",2,"""URS0002A422B0""","""ENSEMBL_GENCODE""",null,"""havana_tagene_homo_sapiens""","[""CLS3 project""]",1,"""FCHS_110516""","""chr13""",31161745,"""31162676""","""cp7@HSPH1""","[""ENST00000954002.1"", ""ENST00000320027.10"", … ""AK311464""]","""0""",0,"[{""EH38E3059049"",""pELS""}, {""EH38E3059050"",""PLS""}, … {""EH38E3059052"",""pELS""}]","[{""RFX1"",4409,[{""SRX2423398"",4409}, {""SRX2424673"",4103}, … {""SRX5457514"",596}]}, {""HSF1"",3960,[{""SRX16646095"",3960}, {""SRX16646102"",3724}, … {""SRX13845688"",503}]}, … {""ZHX2"",507,[{""SRX3636887"",507}]}]","[{""HS_00184065"",""H1 Derived Neuronal Progenitor Cultured Cells"",""Brain"",""Brain"",""SVM""}, {""HS_00184066"",""H1 BMP4 Derived Trophoblast Cultured Cells"",""Trophoblast cell"",""Others"",""SVM""}, … {""HS_00184072"",""Fetal Intestine Small"",""Small intestine"",""Intestine"",""SVM""}]","[{""hg19::chr13:31735993..31736009,-;hg_136690.1"",""p3@HSPH1""}, {""hg19::chr13:31736027..31736053,-;hg_136691.1"",""p1@HSPH1""}, … {""hg19::chr13:31736780..31736803,+;hg_134654.1"",""hg_134654.1""}]",null,-0.981672,"""PLA""","""0,0,204""","""blue""",31161744,31162676,"[{""MA0486.1"",""HSF1"",206,220,""-"",18.2817,3.7000e-7,0.0915,""ATTCTGGAAAGTTCT""}, {""MA0486.2"",""HSF1"",207,219,""-"",17.0,0.000001,0.265,""TTCTGGAAAGTTC""}, … {""MA0486.1"",""HSF1"",291,305,""+"",10.3662,0.0000618,0.501,""CTTCTGGAAAGATTC""}]",9,"[0.075107, 0.067597]","[{31161866,31161936,""chr13""}, {31161876,31161939,""chr13""}]","[1.687655, 3.307098]","[0.000132, 1.7368e-8]"